### Uploading Stata `.dta` Files from Autor, Dorn, Hanson and Creating Silver Spark Tables

In [0]:
# Libraries:
import pandas as pd
import os
import numpy as np

In [0]:
# Loop over files in the /Volumes/workspace/default/china_trade_data/ directory:
# These are prepped .dta files from David Dorn's personal site related to the
# 2013 China Syndrome paper I am replicating:

# Location in databricks:
dir = "/Volumes/workspace/default/china_trade_data/"

# Print file names for clarity of list ordering:
for file in os.listdir(dir):
    print(file)

# Load each file into a pandas df:
raw_dfs = [pd.read_stata(os.path.join(dir, file)) for file in os.listdir(dir)]

In [0]:
# Display one of the dataframes from the load:
raw_dfs[0]

In [0]:
# convert the pandas DFs to spark format:
spark_dfs = [spark.createDataFrame(_df) for _df in raw_dfs]
display(spark_dfs[0])

### Data Profiling:

In [0]:
# check for null values in each column:
from pyspark.sql.functions import col, sum

spark_df = spark_dfs[3] # use the trade dataset for this example
null_counts = spark_df.select(
    [sum(col(c).isNull().cast('int')).alias(c) for c in spark_df.columns]
)
display(null_counts)

In [0]:
# profile the importer column:
display(spark_df.select('importer').distinct())

### Practicing Data Aggregation/Validation:

In [0]:
# practice spark data manipulation syntax:
display(
    spark_df
    .filter(spark_df.importer == 'USA')
    .groupBy('importer', 'exporter')
    .sum('imports')
    .orderBy('sum(imports)', ascending=False)
)

### Create Silver Delta Tables with Spark:

In [0]:
# Create a new schema for the project:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.china_shock")

# Set the silver delta table names: 
table_names = [
    'fig1_data',
    'sic87dd_trade_data',
]

# Create a silver delta table for each of the pandas dataframes:
for _df, table_name in zip(spark_dfs, table_names):
    _df.write.format('delta').mode('overwrite').option(
        'overwriteSchema', True
    ).saveAsTable(
        f'workspace.china_shock.{table_name}'
    )


workspace